In [1]:
# Mô tả: Cấu hình biến môi trường và số luồng cho BLAS/TF
import os

os.environ['OPENBLAS_NUM_THREADS'] = '44'  # 50% cores
os.environ['MKL_NUM_THREADS'] = '44'
os.environ['OMP_NUM_THREADS'] = '44'
os.environ['NUMEXPR_NUM_THREADS'] = '44'

# TensorFlow threading
os.environ['TF_NUM_INTRAOP_THREADS'] = '44'  # Parallel ops
os.environ['TF_NUM_INTEROP_THREADS'] = '8'   # Independent ops

# turn off oneDNN optimization if needed
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

print("Configured for 88-core CPU")

Configured for 88-core CPU


In [2]:
from pathlib import Path

# 03 đã chuyển từ `Code/` vào `Code/Code_SDC_V1/`. Dò bằng glob thay vì hardcode một
# đường dẫn nữa, để lần chuyển thư mục sau không làm notebook này chết lần nữa.
_cwd = Path.cwd().resolve()
_NB03 = next((hit for p in (_cwd, *_cwd.parents)
              for hit in sorted((p / "Code").glob("**/03_train_model.ipynb"))), None)
assert _NB03 is not None, f"Không thấy 03_train_model.ipynb dưới Code/ quanh {_cwd}"

if not globals().get("SDC_DEFS_LOADED"):
    SDC_IMPORT_ONLY = True
    try:
        get_ipython().run_line_magic("run", f'-i "{_NB03}"')
    finally:
        del SDC_IMPORT_ONLY

import json
from datetime import datetime
from collections import Counter, OrderedDict

import joblib
import numpy as np
import pandas as pd
from IPython.display import display


Configured for 88-core CPU
Gốc dự án: C:\Users\admin\Desktop\02.SDC


Đã nạp hàm SDC từ 03_train_model.ipynb

## Tập train: CHỈ 50 dòng capture FIELD — CIC-2022 đã bỏ hẳn

Danh sách thiết bị FIELD lấy động từ cột `scenario` — không hardcode tên, để notebook
này vẫn đúng khi bạn capture thêm thiết bị/ngày FIELD mới.

**Đổi so với bản trước:** 8139 dòng CIC-2022 (IDLE + POWER) trước đây làm nền
`__unknown__` để dạy rừng cách *từ chối*. Chúng đã bị loại hoàn toàn: không làm lớp
thật, không làm nền, không góp vocab TF-IDF. Hệ quả trực tiếp — `__unknown__` biến mất
khỏi `classes_`, rừng thành closed-set thuần: mọi input đều rơi vào một trong các nhãn
FIELD, không bao giờ tự nói "không biết". Khả năng từ chối chuyển hết sang **ngưỡng
abstain** của `sdc_iden.onnx` (notebook 08), là thứ hiện chưa hiệu chỉnh.

In [3]:
UNKNOWN_LABEL = "__unknown__"      # vẫn giữ tên: `sdc_iden.onnx` dùng nó làm nhãn abstain

sessions_all, feature_groups = load_sessions(SESSIONS_PATH)
features = feature_groups["all"]

# Lọc ngay ở đây, một chỗ. Mọi thứ phía dưới (encoder, forest, đánh giá, ONNX) chỉ nhìn
# thấy `sessions`, nên không còn đường nào để dòng CIC-2022 lọt vào tập train.
sessions = sessions_all[sessions_all["scenario"].eq("FIELD")].reset_index(drop=True)
FIELD_DEVICES = sorted(sessions["canonical_device"].unique())
n_dropped = len(sessions_all) - len(sessions)

print(f"Train trên {len(sessions)} dòng / {len(FIELD_DEVICES)} thiết bị FIELD — "
      f"bỏ {n_dropped} dòng ngoài FIELD "
      f"({dict(sessions_all.loc[~sessions_all.scenario.eq('FIELD'), 'scenario'].value_counts())})")
for head in LABEL_COLS:
    vc = sessions[head].value_counts()
    print(f"  {head}: {len(vc)} lớp — {dict(vc)}")

Train trên 50 dòng / 12 thiết bị FIELD — bỏ 8139 dòng ngoài FIELD ({'IDLE': np.int64(8022), 'POWER': np.int64(117)})

  make: 8 lớp — {'Generic Laptop': np.int64(22), 'Camera': np.int64(6), 'Linova/Linux': np.int64(6), 'Samsung': np.int64(6), 'Raspberry Pi': np.int64(3), 'Xiaomi': np.int64(3), 'Apple': np.int64(3), 'OPPO': np.int64(1)}

  type: 4 lớp — {'Laptop': np.int64(28), 'Smartphone': np.int64(13), 'IP Camera': np.int64(6), 'Single-board Computer': np.int64(3)}

  model: 11 lớp — {'Windows Desktop DELL': np.int64(11), 'Generic IP Camera': np.int64(6), 'Windows Laptop HP': np.int64(6), 'Linova Laptop HP': np.int64(6), 'Samsung Galaxy': np.int64(6), 'Windows Desktop HP': np.int64(5), 'Raspberry Pi': np.int64(3), 'iPhone': np.int64(3), 'Redmi Note 14 Pro': np.int64(2), 'OPPO A92': np.int64(1), 'Redmi Note 10': np.int64(1)}

## Fit encoder + model (250 cây, giống cấu hình đang dùng ở `03`)

Encoder giờ fit trên **đúng 50 dòng FIELD**. Trước đây nó fit trên cả corpus vì lập luận
"vocab phải phủ được nền, không thì token CIC-2022 thành out-of-vocab hết và *trông khác
lạ* mất nghĩa". Lập luận đó chết cùng với nền: không còn nền thì không còn gì phải phủ.

Đổi lại được đúng cái mà hai FIX của bản trước phải đi vòng mới chữa được — vocab TF-IDF
không còn phải tranh chỗ với 8139 dòng CIC-2022, nên token nhận dạng của 12 thiết bị
FIELD giữ được hết mà **không cần** bỏ trần `max_features` nữa (vẫn bỏ, vì 50 dòng thì
trần không có tác dụng gì). Kích thước input tensor tụt từ 1088 xuống **445** feature:

| cột | vocab cũ (fit cả corpus) | vocab mới (fit 50 dòng FIELD) |
|---|---|---|
| `dhcp_vci` | 40 | 7 |
| `dns_tokens` | 622 | 175 |
| `mdns_tokens` | 74 | 40 |
| `tls_sni_tokens` | 312 | 187 |

Những token từng bị đá ra (`detectportal` `safebrowsing` `archlinux` `vntek` `vnpt`…)
giờ nằm trong vocab vì chúng là token **duy nhất** còn tồn tại.

Cái này **không** chữa được, y như bản trước: lớp chỉ có 1–6 phiên, tất cả từ MỘT ngày
capture và MỘT MAC. Xem cell dưới.

In [4]:
# --- Bỏ trần `max_features` của TF-IDF -------------------------------------------
# `_make_vectorizer` ở 03 đọc `TFIDF_MAX_FEATURES` tại thời điểm fit, nên ghi đè ở đây là
# đủ — không đụng vào contract `sdc-tiered-v2` mà 03/05/06/07 đang dùng chung.
#
# Với 50 dòng thì trần 200/100/150/100 gần như không cắt gì nữa (vocab thật là
# 175/40/187/7). Vẫn bỏ trần để con số vocab là *toàn bộ token FIELD*, không phải
# "toàn bộ trừ chỗ nào chạm trần" — một biến ít đi khi sau này capture thêm ngày.
TFIDF_MAX_FEATURES = {col: None for col in TFIDF_MAX_FEATURES}

encoder = fit_encoder(sessions, features)
X_all, feature_names, _ = apply_encoder(sessions, encoder)
print(f"X = {X_all.shape} — vocab: "
      + ", ".join(f"{c}={len(encoder['ct'].named_transformers_['t_' + c].vocabulary_)}"
                  for c in encoder["text_cols"]))

# --- Train: nhãn thật, không nền -------------------------------------------------
# Bản trước phải dựng `y = np.where(is_field, nhãn, UNKNOWN_LABEL)` rồi loại tiếp những
# dòng CIC-2022 trùng nhãn thật của một lớp FIELD (head `type` dính 2471 dòng IP Camera
# chọi 6 dòng FIELD). Cả hai thứ đó biến mất: `sessions` chỉ còn FIELD nên `y` là nhãn
# thật, và không có dòng nào để đụng độ.
models = {}
train_rows = {}
for head in LABEL_COLS:
    y = sessions[head].astype(str).to_numpy()
    m = make_model()
    m.fit(X_all, y)
    models[head] = m
    train_rows[head] = int(len(y))
    print(f"  fit {head}: {len(m.classes_)} lop, {len(y)} dong train — {list(m.classes_)}")

# --- Cái này KHÔNG chữa được gì ----------------------------------------------------
# 1. Không còn lớp `__unknown__`. Rừng không có cách nào trả lời "thiết bị lạ" nữa —
#    argmax luôn rơi vào một nhãn FIELD. Đo trên chính 8139 dòng CIC-2022 vừa bỏ: ở mốc
#    conf >= 0.60, head `type` báo nhầm 4861/8139 dòng (chủ yếu thành `Smartphone`),
#    `make` 2384/8139, `model` 2389/8139. Trước khi bỏ CIC thì con số này là 0/8139.
#    Việc từ chối giờ hoàn toàn nằm ở ngưỡng abstain của `sdc_iden.onnx`.
# 2. Lớp dưới `RARE_THRESHOLD` vẫn là ghi nhớ thuộc lòng, không phải khái quát.
# 3. Linova Laptop vẫn không có feature nội tại nào khác 4 laptop FIELD còn lại — cùng
#    `tls_fp`, không mDNS, không `dhcp_vci`, `dhcp_prl` trùng hệt Raspberry Pi. Thứ duy
#    nhất tách nó ra là tập token DNS/SNI của người dùng.
print(f"\nSo phien moi lop (nhac lai — RARE_THRESHOLD={RARE_THRESHOLD}):")
for head in LABEL_COLS:
    thin = sessions[head].value_counts()
    thin = thin[thin < RARE_THRESHOLD]
    print(f"  {head}: {dict(thin)}")

X = (50, 445) — vocab: dhcp_vci=7, dns_tokens=175, mdns_tokens=40, tls_sni_tokens=187

  fit make: 8 lop, 50 dong train — ['Apple', 'Camera', 'Generic Laptop', 'Linova/Linux', 'OPPO', 'Raspberry Pi', 'Samsung', 'Xiaomi']

  fit type: 4 lop, 50 dong train — ['IP Camera', 'Laptop', 'Single-board Computer', 'Smartphone']

  fit model: 11 lop, 50 dong train — ['Generic IP Camera', 'Linova Laptop HP', 'OPPO A92', 'Raspberry Pi', 'Redmi Note 10', 'Redmi Note 14 Pro', 'Samsung Galaxy', 'Windows Desktop DELL', 'Windows Desktop HP', 'Windows Laptop HP', 'iPhone']


So phien moi lop (nhac lai — RARE_THRESHOLD=10):

  make: {'Camera': np.int64(6), 'Linova/Linux': np.int64(6), 'Samsung': np.int64(6), 'Raspberry Pi': np.int64(3), 'Xiaomi': np.int64(3), 'Apple': np.int64(3), 'OPPO': np.int64(1)}

  type: {'IP Camera': np.int64(6), 'Single-board Computer': np.int64(3)}

  model: {'Generic IP Camera': np.int64(6), 'Windows Laptop HP': np.int64(6), 'Linova Laptop HP': np.int64(6), 'Samsung Galaxy': np.int64(6), 'Windows Desktop HP': np.int64(5), 'Raspberry Pi': np.int64(3), 'iPhone': np.int64(3), 'Redmi Note 14 Pro': np.int64(2), 'OPPO A92': np.int64(1), 'Redmi Note 10': np.int64(1)}

## Đóng gói + lưu

Định dạng riêng `sdc-closedset-v1` — **không tương thích** với `Predictor` (contract
`sdc-tiered-v2`) ở `03`/`06`: không có bảng L1, không có ngưỡng theo `n_sources`.

`unknown_label` vẫn nằm trong bundle vì `sdc_iden.onnx` dùng đúng chuỗi đó làm nhãn
abstain, **nhưng nó không còn là một lớp của forest** — `predict_proba` không bao giờ
sinh ra nó. Cơ chế từ chối đã chuyển hẳn sang ngưỡng ở notebook 08.

In [5]:
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_field_closedset")
run_dir = MODELS / RUN_ID
run_dir.mkdir(parents=True)

bundle = {
    "format": "sdc-closedset-v1",
    # Giữ khoá để 08 và `ClosedSetPredictor` đọc được, nhưng đây là nhãn ABSTAIN chứ
    # không còn là một lớp forest — xem markdown trên.
    "unknown_label": UNKNOWN_LABEL,
    "heads": list(LABEL_COLS),
    "feature_cols": features,
    "feature_names": feature_names,
    "encoder": encoder,
    "models": models,
    "field_devices": FIELD_DEVICES,
    "trained_on": "FIELD",
}
joblib.dump(bundle, run_dir / "model.joblib", compress=3)
(run_dir / "meta.json").write_text(json.dumps({
    "format": "sdc-closedset-v1",
    "run_id": RUN_ID,
    "created": datetime.now().isoformat(timespec="seconds"),
    "dataset": str(SESSIONS_PATH),
    "trained_on": "FIELD",
    "n_sessions_total": int(len(sessions_all)),
    "n_sessions_train": int(len(sessions)),
    "n_sessions_dropped": int(n_dropped),
    "n_field_devices": len(FIELD_DEVICES),
    "field_devices": FIELD_DEVICES,
    "heads": {h: int(len(models[h].classes_)) for h in LABEL_COLS},
    "labels": {h: [str(c) for c in models[h].classes_] for h in LABEL_COLS},
    "train_rows": train_rows,
    "n_features": int(X_all.shape[1]),
    "tfidf_max_features": TFIDF_MAX_FEATURES,
    "note": ("Train CHỈ trên 50 dòng capture FIELD. CIC-2022 (8139 dòng IDLE+POWER) đã "
             "bỏ hẳn: không làm lớp thật, không làm nền __unknown__, không góp vocab "
             "TF-IDF. Encoder fit trên 50 dòng FIELD -> 445 feature (trước: 1088). "
             "HỆ QUẢ ĐÃ ĐO: forest không còn lớp __unknown__ nên không tự từ chối được "
             "thiết bị lạ; ở mốc conf>=0.60 nó gán nhãn FIELD cho 4861/8139 dòng CIC-2022 "
             "ở head type, 2384 ở make, 2389 ở model (trước khi bỏ CIC: 0/8139). Việc từ "
             "chối nằm hoàn toàn ở ngưỡng abstain của sdc_iden.onnx, hiện CHƯA hiệu chỉnh. "
             "CẢNH BÁO CHƯA GỠ: đánh giá FIELD là in-sample, 1 ngày capture, 1 MAC mỗi "
             "thiết bị. Các lớp dưới RARE_THRESHOLD=10 (Raspberry Pi 3 phiên, iPhone 3, "
             "Redmi Note 14 Pro 2, OPPO A92 1, Redmi Note 10 1) là ghi nhớ thuộc lòng. "
             "Linova Laptop không có feature nội tại nào khác 4 laptop FIELD còn lại."),
}, indent=2, ensure_ascii=False), encoding="utf-8")

print("Đã lưu:", run_dir)

Đã lưu:

C:\Users\admin\Desktop\02.SDC\Models\20260916_152342_field_closedset

## Predictor tối giản cho closed-set

Chỉ argmax `predict_proba`. `is_unknown` giữ lại cho tương thích chữ ký nhưng **luôn
`False`** — `__unknown__` không còn là lớp của forest.

In [6]:
class ClosedSetPredictor:
    def __init__(self, run_dir):
        bundle = joblib.load(Path(run_dir) / "model.joblib")
        assert bundle["format"] == "sdc-closedset-v1"
        # Nhãn abstain của sdc_iden.onnx, KHÔNG phải một lớp của forest nữa:
        # `is_unknown` dưới đây luôn False. Giữ lại cho tương thích chữ ký.
        self.unknown_label = bundle["unknown_label"]
        self.heads = bundle["heads"]
        self.feature_cols = bundle["feature_cols"]
        self.encoder = bundle["encoder"]
        self.models = bundle["models"]
        self.field_devices = bundle["field_devices"]

    def predict_row(self, row):
        frame = pd.DataFrame([row])[self.feature_cols]
        X, _, _ = apply_encoder(frame, self.encoder)
        out = {}
        for head in self.heads:
            proba = self.models[head].predict_proba(X)[0]
            classes = self.models[head].classes_
            idx = proba.argmax()
            top1, conf = classes[idx], float(proba[idx])
            out[head] = {"top1": top1, "confidence": conf,
                         "is_unknown": top1 == self.unknown_label}
        return out

    def predict_device(self, rows):
        """Gộp nhiều phiên/cửa sổ của cùng một thiết bị bằng bỏ phiếu đa số trên top1
        (đơn giản hơn DeviceTracker ở 03 — không có ratio/ambiguous; từ chối là việc
        của ngưỡng abstain ở 08, không phải của predictor này)."""
        votes = {h: Counter() for h in self.heads}
        for row in rows:
            out = self.predict_row(row)
            for head in self.heads:
                votes[head][out[head]["top1"]] += 1
        result = {}
        for head in self.heads:
            top1, n = votes[head].most_common(1)[0]
            result[head] = {"top1": top1, "n_votes": n, "n_total": len(rows),
                             "is_unknown": top1 == self.unknown_label}
        return result


predictor = ClosedSetPredictor(run_dir)
print("Nạp lại từ", run_dir, "— OK")

Nạp lại từ

C:\Users\admin\Desktop\02.SDC\Models\20260916_152342_field_closedset

— OK

## Đánh giá

1. **Tự soi FIELD** — in-sample, chỉ để kiểm tra model học được (không phải bằng chứng
   khái quát, xem cảnh báo ở đầu notebook).
2. **IoT Sentinel** — 29 thiết bị hoàn toàn chưa đưa vào lúc fit. Trước đây đây là phép
   thử "từ chối đúng không". Giờ forest không có nhãn để từ chối, nên nó đo thứ khác:
   **rừng tự tin đến đâu khi bị hỏi về thiết bị nó chưa từng thấy** — tức là ngưỡng
   abstain sẽ phải gánh nặng cỡ nào.
3. **Biên độ** — top1 trừ top2, xem thắng có sát nút không.

In [7]:
# Mốc tham khảo để đọc con số "báo nhầm". Ngưỡng THẬT nằm ở `thresholds` của
# `sdc_iden.onnx` (notebook 08) và phụ thuộc `n_sources`; đây chỉ là một lát cắt.
REJECT_AT = 0.60


def eval_rows(sub_frame, label, is_field_data):
    Xs, _, _ = apply_encoder(sub_frame, predictor.encoder)
    print(f"\n=== {label} ({len(sub_frame)} dòng) ===")
    for head in predictor.heads:
        proba = predictor.models[head].predict_proba(Xs)
        classes = predictor.models[head].classes_
        top_idx = proba.argmax(axis=1)
        top1 = classes[top_idx]
        conf = proba[np.arange(len(top_idx)), top_idx]
        if is_field_data:
            truth = sub_frame[head].astype(str).to_numpy()
            correct = int((top1 == truth).sum())
            print(f"  {head:6s} đúng: {correct:4d}/{len(sub_frame)}   "
                  f"sai: {len(sub_frame) - correct:4d}   "
                  f"conf trung vị {np.median(conf):.3f}")
        else:
            # Không còn lớp `__unknown__` để từ chối bằng NHÃN. Mọi dòng đều bị gán vào
            # một thiết bị FIELD, nên phép đo duy nhất còn lại là confidence.
            over = int((conf >= REJECT_AT).sum())
            common = dict(pd.Series(top1).value_counts().head(3))
            print(f"  {head:6s} conf trung vị {np.median(conf):.3f}  "
                  f"p90 {np.percentile(conf, 90):.3f}  "
                  f"BÁO NHẦM ở mốc {REJECT_AT}: {over:4d}/{len(sub_frame)}")
            print(f"         bị gán nhiều nhất: {common}")


eval_rows(sessions, "FIELD tự soi (in-sample — xem cảnh báo ở đầu notebook)",
          is_field_data=True)


=== FIELD tự soi (in-sample — xem cảnh báo ở đầu notebook) (50 dòng) ===

  make   đúng:   50/50   sai:    0   conf trung vị 0.844

  type   đúng:   50/50   sai:    0   conf trung vị 0.928

  model  đúng:   50/50   sai:    0   conf trung vị 0.734

In [8]:
def load_windows(path):
    by_device = OrderedDict()
    with path.open(encoding="utf-8") as fh:
        for line in fh:
            w = json.loads(line)
            by_device.setdefault(w["device"], []).append(w)
    return by_device


SENTINEL_OUT = ROOT / "test_model" / "data_test" / "out"
by_device = load_windows(SENTINEL_OUT / "windows.jsonl")

rows = []
for device, windows in by_device.items():
    for w in windows:
        row = aggregate(w["records"], features)
        rows.append(row)
sentinel_frame = pd.DataFrame(rows)

eval_rows(sentinel_frame,
          f"IoT Sentinel ({len(by_device)} thiết bị lạ, chưa từng đưa vào lúc fit)",
          is_field_data=False)

# Chính tập vừa bị loại khỏi train — phép đo trực tiếp cho cái giá của việc bỏ CIC-2022.
eval_rows(sessions_all[~sessions_all["scenario"].eq("FIELD")],
          "CIC-2022 (8139 dòng vừa bỏ khỏi tập train — trước đây là nền __unknown__)",
          is_field_data=False)


=== IoT Sentinel (29 thiết bị lạ, chưa từng đưa vào lúc fit) (510 dòng) ===

  make   conf trung vị 0.680  p90 0.772  BÁO NHẦM ở mốc 0.6:  280/510

         bị gán nhiều nhất: {'Camera': np.int64(433), 'Samsung': np.int64(65), 'Xiaomi': np.int64(12)}

  type   conf trung vị 0.648  p90 0.764  BÁO NHẦM ở mốc 0.6:  288/510

         bị gán nhiều nhất: {'IP Camera': np.int64(280), 'Smartphone': np.int64(230)}

...

         bị gán nhiều nhất: {'Camera': np.int64(4342), 'Samsung': np.int64(3179), 'Xiaomi': np.int64(618)}

  type   conf trung vị 0.656  p90 0.824  BÁO NHẦM ở mốc 0.6: 4861/8139

         bị gán nhiều nhất: {'Smartphone': np.int64(5757), 'IP Camera': np.int64(2382)}

  model  conf trung vị 0.392  p90 0.836  BÁO NHẦM ở mốc 0.6: 2389/8139

         bị gán nhiều nhất: {'Generic IP Camera': np.int64(5014), 'Samsung Galaxy': np.int64(2313), 'Redmi Note 14 Pro': np.int64(812)}

In [9]:
def margin_report(sub_frame, label):
    """top1 - top2. Thay cho `p(__unknown__)` của bản trước — lớp đó không còn tồn tại."""
    Xs, _, _ = apply_encoder(sub_frame, predictor.encoder)
    print(f"\n=== Biên độ top1 − top2 — {label} ===")
    for head in predictor.heads:
        proba = np.sort(predictor.models[head].predict_proba(Xs), axis=1)
        top1, margin = proba[:, -1], proba[:, -1] - proba[:, -2]
        print(f"  {head:6s} top1: mean={top1.mean():.3f} max={top1.max():.3f}   "
              f"margin: min={margin.min():.3f} mean={margin.mean():.3f}")


margin_report(sentinel_frame, "IoT Sentinel (ngoài list)")
margin_report(sessions, "FIELD (trong list, in-sample)")


=== Biên độ top1 − top2 — IoT Sentinel (ngoài list) ===

  make   top1: mean=0.539 max=0.796   margin: min=0.020 mean=0.390

  type   top1: mean=0.604 max=0.788   margin: min=0.004 mean=0.355

  model  top1: mean=0.546 max=0.820   margin: min=0.000 mean=0.425


=== Biên độ top1 − top2 — FIELD (trong list, in-sample) ===

  make   top1: mean=0.818 max=1.000   margin: min=0.168 mean=0.735

  type   top1: mean=0.902 max=1.000   margin: min=0.164 mean=0.823

  model  top1: mean=0.744 max=0.996   margin: min=0.136 mean=0.638

## Xuất ONNX — một input / một output

In [10]:
import warnings

import onnx
import onnxruntime as ort
from onnx import TensorProto as TP
from onnx import compose, helper
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType, StringTensorType

ONNX_INPUT = "input"
ONNX_OUTPUT = "output"
ONNX_FILE = "sdc_closedset.onnx"
ONNX_CONTRACT_VERSION = "1.0.0"

ONNX_OPSET = 20
ONNX_ML_OPSET = 3

# Xem giải thích đầy đủ ở 07_export_onnx.ipynb (mục "Lắp ráp"): thiếu locale thì
# onnxruntime dựng `en_US.UTF-8` lúc nạp model và chết trên musl (router OpenWrt).
ONNX_LOCALE = "C"

### Bộ dựng graph + front

Rút gọn từ `OnnxGraphBuilder` ở 07 — chỉ giữ `name`/`add`/`const`, bỏ `label_encode`/
`concat_str` vì không cần dựng khoá L1 ở đây.

In [11]:
class OnnxGraphBuilder:
    """Gom node + initializer cho một graph, tự sinh tên không đụng nhau."""

    def __init__(self, prefix="sdc"):
        self.prefix = prefix
        self.nodes = []
        self.inits = []
        self._n = 0

    def name(self, stem):
        self._n += 1
        return f"{self.prefix}/{stem}_{self._n}"

    def add(self, op_type, inputs, out_stem, **attrs):
        out = self.name(out_stem)
        domain = attrs.pop("domain", "")
        self.nodes.append(helper.make_node(op_type, list(inputs), [out],
                                           name=self.name(f"n_{op_type}"),
                                           domain=domain, **attrs))
        return out

    def const(self, array, stem, dtype=None):
        out = self.name(stem)
        if dtype == TP.STRING:
            values = np.asarray(array, dtype=object)
            self.inits.append(helper.make_tensor(
                out, TP.STRING, list(values.shape),
                [s.encode("utf-8") for s in values.ravel()]))
        else:
            self.inits.append(onnx.numpy_helper.from_array(np.asarray(array), out))
        return out


def onnx_build_front(gb, cols, num_cols):
    """`input` string [N,len(cols)] -> dict {tên cột: tensor}. Cột numeric Cast sang float32."""
    outs = [gb.name(f"col/{c}") for c in cols]
    gb.nodes.append(helper.make_node("Split", [ONNX_INPUT], outs,
                                     name=gb.name("n_Split"), axis=1,
                                     num_outputs=len(cols)))
    num = set(num_cols)
    return {col: (gb.add("Cast", [t], f"num/{col}", to=TP.FLOAT) if col in num else t)
            for col, t in zip(cols, outs)}


def onnx_relink(graph, rename):
    """Sao node + initializer của một subgraph, đổi tên input theo `rename`."""
    nodes = []
    for node in graph.node:
        copy = onnx.NodeProto()
        copy.CopyFrom(node)
        for i, name in enumerate(copy.input):
            if name in rename:
                copy.input[i] = rename[name]
        nodes.append(copy)
    return nodes, list(graph.initializer)

### Chuyển encoder + 3 rừng cây, ghi locale, cắt trọng số

`onnx_sub_models`, `onnx_prune`, `onnx_set_locale`/`onnx_assert_locale` giống hệt 07 —
không phụ thuộc contract tiered, chỉ cần `bundle["encoder"]`/`["models"]`/
`["feature_names"]`, những khoá bundle closed-set này cũng có.

In [12]:
def onnx_sub_models(bundle):
    enc = bundle["encoder"]
    initial = ([(c, FloatTensorType([None, 1])) for c in enc["num_cols"]]
               + [(c, StringTensorType([None, 1]))
                  for c in enc["cat_cols"] + enc["text_cols"]])
    ct = convert_sklearn(enc["ct"], "sdc_encoder", initial_types=initial,
                         target_opset=ONNX_OPSET)
    ct = compose.add_prefix(ct, "enc/", rename_inputs=False)

    n_feat = len(bundle["feature_names"])
    forests = {}
    for head in bundle["heads"]:
        clf = bundle["models"][head]
        m = convert_sklearn(clf, f"sdc_{head}",
                            initial_types=[("X", FloatTensorType([None, n_feat]))],
                            target_opset=ONNX_OPSET,
                            options={id(clf): {"zipmap": False}})
        forests[head] = compose.add_prefix(m, f"rf_{head}/")
    return ct, forests


def onnx_prune(model):
    """Bỏ trọng số lá bằng 0 và hai thuộc tính đang mang đúng giá trị mặc định."""
    stats = {"weights_kept": 0, "weights_dropped": 0, "attrs_dropped": []}
    for node in model.graph.node:
        if node.op_type not in ("TreeEnsembleClassifier", "TreeEnsembleRegressor"):
            continue
        att = {a.name: a for a in node.attribute}
        weights = np.asarray(att["class_weights"].floats, dtype=np.float32)
        keep = np.flatnonzero(weights != 0)
        stats["weights_kept"] += int(keep.size)
        stats["weights_dropped"] += int(weights.size - keep.size)
        for field in ("class_ids", "class_nodeids", "class_treeids"):
            values = [int(v) for v in np.asarray(att[field].ints)[keep]]
            del att[field].ints[:]
            att[field].ints.extend(values)
        values = [float(v) for v in weights[keep]]
        del att["class_weights"].floats[:]
        att["class_weights"].floats.extend(values)

        hit = att.get("nodes_hitrates")
        miss = att.get("nodes_missing_value_tracks_true")
        drop = ([("nodes_hitrates", hit)] if hit is not None
                and all(v == 1.0 for v in hit.floats) else [])
        drop += ([("nodes_missing_value_tracks_true", miss)] if miss is not None
                 and all(v == 0 for v in miss.ints) else [])
        for name, attr in drop:
            node.attribute.remove(attr)
            stats["attrs_dropped"].append(name)

    model.graph.ClearField("doc_string")
    for node in model.graph.node:
        node.ClearField("doc_string")
    stats["attrs_dropped"] = sorted(set(stats["attrs_dropped"]))
    return stats


def onnx_set_locale(model, locale=ONNX_LOCALE):
    patched = []
    for node in model.graph.node:
        if node.op_type != "StringNormalizer":
            continue
        for existing in [a for a in node.attribute if a.name == "locale"]:
            node.attribute.remove(existing)
        node.attribute.append(helper.make_attribute("locale", locale))
        patched.append(node.name)
    return patched


def onnx_assert_locale(model, locale=ONNX_LOCALE):
    for node in model.graph.node:
        if node.op_type != "StringNormalizer":
            continue
        got = {a.name: a for a in node.attribute}.get("locale")
        assert got is not None, f"{node.name}: thiếu thuộc tính locale"
        assert got.s.decode() == locale, (
            f"{node.name}: locale {got.s.decode()!r}, cần {locale!r}")

### Lắp graph

Không có `onnx_build_l1`/`onnx_build_head`/`onnx_build_hierarchy` như 07 — mỗi forest tự
trả nhãn cuối, `Reshape` rồi `Concat` là xong. Không còn `__unknown__` trong số nhãn đó.

In [13]:
def onnx_build_model(bundle, prune=True):
    """Dựng ModelProto hoàn chỉnh. Trả (model, thứ tự cột, thống kê tối ưu)."""
    enc = bundle["encoder"]
    cols = enc["num_cols"] + enc["cat_cols"] + enc["text_cols"]
    ct, forests = onnx_sub_models(bundle)

    gb = OnnxGraphBuilder()
    columns = onnx_build_front(gb, cols, enc["num_cols"])

    # rename_inputs=False cua compose.add_prefix van gan tien to trong ban skl2onnx nay,
    # nen noi theo VI TRI (cung thu tu voi initial_types = cols) thay vi theo ten.
    nodes, inits = onnx_relink(ct.graph, {i.name: columns[c] for i, c in zip(ct.graph.input, cols)})
    gb.nodes += nodes
    gb.inits += inits
    features = ct.graph.output[0].name

    final = []
    for head in bundle["heads"]:
        forest = forests[head]
        nodes, inits = onnx_relink(forest.graph, {forest.graph.input[0].name: features})
        gb.nodes += nodes
        gb.inits += inits
        raw_label = forest.graph.output[0].name    # zipmap=False -> (label, probabilities)
        label = gb.add("Reshape", [raw_label, gb.const(np.int64([-1, 1]), "shape")],
                       f"out/{head}")
        final.append(label)

    gb.nodes.append(helper.make_node("Concat", final, [ONNX_OUTPUT],
                                     name=gb.name("n_Concat"), axis=1))

    graph = helper.make_graph(
        gb.nodes, "sdc_closedset",
        [helper.make_tensor_value_info(ONNX_INPUT, TP.STRING, [None, len(cols)])],
        [helper.make_tensor_value_info(ONNX_OUTPUT, TP.STRING, [None, len(bundle["heads"])])],
        gb.inits)

    opsets = {"": ONNX_OPSET, "ai.onnx.ml": ONNX_ML_OPSET}
    for sub in (ct, *forests.values()):
        for imp in sub.opset_import:
            opsets[imp.domain] = max(opsets.get(imp.domain, 0), imp.version)
    model = helper.make_model(
        graph, opset_imports=[helper.make_opsetid(d, v) for d, v in opsets.items()])
    model.ir_version = 10
    model.doc_string = ""

    stats = onnx_prune(model) if prune else {}
    stats["locale_nodes"] = onnx_set_locale(model)
    onnx_assert_locale(model)
    onnx.checker.check_model(model)
    return model, cols, stats


def onnx_contract(bundle, cols, run_id, stats, payload):
    enc = bundle["encoder"]
    return {
        "format": "sdc-closedset-onnx-v1",
        "contract_version": ONNX_CONTRACT_VERSION,
        "run_id": run_id,
        "file": ONNX_FILE,
        "bytes": len(payload),
        "runtime": {"required": "onnxruntime",
                    "reason": "TF-IDF dùng com.microsoft.Tokenizer",
                    "opset": ONNX_OPSET, "ai.onnx.ml": ONNX_ML_OPSET,
                    "string_normalizer_locale": ONNX_LOCALE},
        "io": {
            "input": {"name": ONNX_INPUT, "type": "string", "shape": ["N", len(cols)],
                      "columns": cols,
                      "note": ("Cột numeric gửi dưới dạng chuỗi số ('1', '0'); nguồn "
                               "vắng mặt -> numeric '0', cat/text '<missing>'.")},
            "output": {"name": ONNX_OUTPUT, "type": "string",
                      "shape": ["N", len(bundle["heads"])],
                      "heads": list(bundle["heads"]),
                      "unknown_label": bundle["unknown_label"],
                      "note": ("__unknown__ là một lớp forest tự trả lời — không có tầng "
                               "policy nào ngoài graph quyết định việc này.")},
        },
        "labels": {h: [str(c) for c in bundle["models"][h].classes_] for h in bundle["heads"]},
        "field_devices": bundle["field_devices"],
        "columns": {"num": enc["num_cols"], "cat": enc["cat_cols"], "text": enc["text_cols"]},
        "size_optimization": stats,
    }


def onnx_export(run_dir, prune=True):
    run_dir = Path(run_dir)
    bundle = joblib.load(run_dir / "model.joblib")
    fmt = bundle["format"]
    assert fmt == "sdc-closedset-v1", f"format lạ: {fmt}"
    model, cols, stats = onnx_build_model(bundle, prune=prune)
    payload = model.SerializeToString()
    path = run_dir / ONNX_FILE
    path.write_bytes(payload)
    (run_dir / "contract.json").write_text(
        json.dumps(onnx_contract(bundle, cols, run_dir.name, stats, payload),
                   indent=2, ensure_ascii=False), encoding="utf-8")
    return path

### Xuất candidate

In [14]:
onnx_path = onnx_export(run_dir)
onnx_contract_doc = json.loads((run_dir / "contract.json").read_text(encoding="utf-8"))

_raw, _, _ = onnx_build_model(bundle, prune=False)
_raw_bytes = len(_raw.SerializeToString())
_stats = onnx_contract_doc["size_optimization"]

print(f"run             {run_dir.name}")
print(f"model.joblib    {(run_dir / 'model.joblib').stat().st_size:>9,} bytes")
print(f"onnx chưa cắt   {_raw_bytes:>9,} bytes")
print(f"onnx đã cắt     {onnx_path.stat().st_size:>9,} bytes  "
      f"(-{1 - onnx_path.stat().st_size / _raw_bytes:.1%})")
print(f"trọng số lá giữ {_stats['weights_kept']:,} / "
      f"{_stats['weights_kept'] + _stats['weights_dropped']:,}")
display(pd.DataFrame([onnx_contract_doc["io"]["input"], onnx_contract_doc["io"]["output"]],
                     index=["input", "output"])[["name", "type", "shape"]])

run             20260916_152342_field_closedset

model.joblib      348,597 bytes

onnx chưa cắt   1,408,145 bytes

onnx đã cắt       539,011 bytes  (-61.7%)

trọng số lá giữ 8,809 / 74,146

,name,type,shape
input,input,string,"[N, 40]"
output,output,string,"[N, 3]"


### Parity với `ClosedSetPredictor`



In [15]:
def onnx_to_input(frame, cols):
    out = frame[cols].copy()
    for col in cols:
        values = out[col]
        out[col] = (values.astype(np.float32).astype(str)
                    if pd.api.types.is_numeric_dtype(values) else values.astype(str))
    return out.to_numpy(dtype=object)


def onnx_reference_closedset(frame, heads):
    """Nhãn tham chiếu từ chính model đã fit — chạy theo LÔ, không lặp từng dòng.

    Bản cũ gọi `predictor.predict_row` một lần cho mỗi dòng: 8189 lần dựng lại DataFrame
    1 dòng, chạy lại cả ColumnTransformer (4 TF-IDF) rồi 3 rừng 250 cây trên đúng 1 dòng.
    Đo được 83.6s cho 300 dòng, tức ~38 phút cho cả tập — toàn bộ chỗ chậm nằm ở đây, ONNX
    không liên quan. `apply_encoder` và `predict_proba` đều nhận cả khung: 0.8s cho 8189
    dòng, nhanh hơn ~2850 lần, cùng nhãn từng ô (xem `onnx_spotcheck_rowwise`).
    """
    X, _, _ = apply_encoder(frame[predictor.feature_cols], predictor.encoder)
    return np.column_stack([
        predictor.models[head].classes_[predictor.models[head].predict_proba(X).argmax(axis=1)]
        for head in heads
    ]).astype(object)


def onnx_spotcheck_rowwise(frame, heads, n=200, seed=0):
    """Giữ lại đúng đường chạy cũ — `predict_row` từng dòng — trên một mẫu nhỏ.

    Gộp lô chỉ tương đương chạy từng dòng nếu `apply_encoder` không phụ thuộc kích thước
    lô. Điều đó đang đúng (TF-IDF đã fit, OrdinalEncoder đã đóng băng vocab) nhưng là giả
    định chứ không phải định luật, nên vẫn kiểm — 200 dòng thay vì 8189.
    """
    sample = frame.sample(min(n, len(frame)), random_state=seed)
    batch = onnx_reference_closedset(sample, heads)
    rowwise = np.array([[predictor.predict_row(record)[head]["top1"] for head in heads]
                        for record in sample.to_dict("records")], dtype=object)
    bad = int((batch != rowwise).sum())
    assert not bad, (f"batch và per-row lệch {bad} ô trên {len(sample)} dòng — "
                     "apply_encoder đang phụ thuộc kích thước lô, KHÔNG được gộp lô")
    return len(sample)


def onnx_verify_closedset(run_dir, contract, limit=None, seed=0):
    cols = contract["io"]["input"]["columns"]
    heads = contract["io"]["output"]["heads"]
    frame = sessions
    if limit and limit < len(frame):
        frame = frame.sample(limit, random_state=seed).reset_index(drop=True)

    session = ort.InferenceSession(str(Path(run_dir) / contract["file"]),
                                   providers=["CPUExecutionProvider"])
    got = session.run(None, {contract["io"]["input"]["name"]:
                             onnx_to_input(frame, cols)})[0]
    want = onnx_reference_closedset(frame, heads)

    rows, mismatch = [], {}
    for i, head in enumerate(heads):
        same = got[:, i] == want[:, i]
        rows.append({"head": head, "n": len(frame), "khớp": float(same.mean()),
                     "lệch": int((~same).sum())})
        if not same.all():
            bad = frame.loc[~same, ["canonical_device", head]].copy()
            bad["onnx"] = got[~same, i]
            bad["predictor"] = want[~same, i]
            mismatch[head] = bad
    return pd.DataFrame(rows).set_index("head"), mismatch


_n_spot = onnx_spotcheck_rowwise(sessions, onnx_contract_doc["io"]["output"]["heads"])
print(f"Spot-check: gộp lô khớp predict_row từng dòng trên {_n_spot} dòng mẫu")

onnx_parity, onnx_mismatch = onnx_verify_closedset(run_dir, onnx_contract_doc)
display(onnx_parity)
for head, bad in onnx_mismatch.items():
    print(f"\n{head}: {len(bad)} dòng lệch")
    display(bad.head(15))
assert not onnx_mismatch, "ONNX và ClosedSetPredictor không khớp — xem bảng lệch ở trên"
print("Parity OK — graph và ClosedSetPredictor cho cùng nhãn trên mọi dòng")

Spot-check: gộp lô khớp predict_row từng dòng trên 50 dòng mẫu

,n,khớp,lệch
head,,,
make,50,1.0,0
type,50,1.0,0
model,50,1.0,0


Parity OK — graph và ClosedSetPredictor cho cùng nhãn trên mọi dòng

In [16]:
def verdict_of(pred, truth):
    return "OK" if pred == truth else "SAI"


rows = []
for device, group in sessions.groupby("canonical_device", sort=True):
    device_rows = [r.to_dict() for _, r in group.iterrows()]
    result = predictor.predict_device(device_rows)
    entry = {"device": device, "n_phien": len(device_rows)}
    for head in predictor.heads:
        truth = group[head].iloc[0]
        pred = result[head]["top1"]
        entry[f"{head}_pred"] = pred
        entry[f"{head}_truth"] = truth
        entry[f"{head}_verdict"] = verdict_of(pred, truth)
    rows.append(entry)

closedset_report = pd.DataFrame(rows)
closedset_summary = pd.DataFrame({
    head: closedset_report[f"{head}_verdict"].value_counts() for head in predictor.heads
}).T.fillna(0).astype(int)
display(closedset_summary)
display(closedset_report)

,OK
make,12
type,12
model,12


,device,n_phien,make_pred,make_truth,make_verdict,type_pred,type_truth,type_verdict,model_pred,model_truth,model_verdict
0,Desktop PC DNGJHRT,5,Generic Laptop,Generic Laptop,OK,Laptop,Laptop,OK,Windows Desktop HP,Windows Desktop HP,OK
1,Desktop PC DucAnh,6,Generic Laptop,Generic Laptop,OK,Laptop,Laptop,OK,Windows Desktop DELL,Windows Desktop DELL,OK
2,IP Camera (field),6,Camera,Camera,OK,IP Camera,IP Camera,OK,Generic IP Camera,Generic IP Camera,OK
3,Laptop VAF70SQ6,6,Generic Laptop,Generic Laptop,OK,Laptop,Laptop,OK,Windows Laptop HP,Windows Laptop HP,OK
4,Lee Kingdom Laptop,5,Generic Laptop,Generic Laptop,OK,Laptop,Laptop,OK,Windows Desktop DELL,Windows Desktop DELL,OK
5,Linova Laptop (linova),6,Linova/Linux,Linova/Linux,OK,Laptop,Laptop,OK,Linova Laptop HP,Linova Laptop HP,OK
6,OPPO A92,1,OPPO,OPPO,OK,Smartphone,Smartphone,OK,OPPO A92,OPPO A92,OK
7,Raspberry Pi,3,Raspberry Pi,Raspberry Pi,OK,Single-board Computer,Single-board Computer,OK,Raspberry Pi,Raspberry Pi,OK
8,Samsung Phone (Duc Anh),6,Samsung,Samsung,OK,Smartphone,Smartphone,OK,Samsung Galaxy,Samsung Galaxy,OK
9,Xiaomi Redmi Note 10,1,Xiaomi,Xiaomi,OK,Smartphone,Smartphone,OK,Redmi Note 10,Redmi Note 10,OK
